# distopt: MUDAG validation notebook

This notebook is meant for **mathematical / numerical sanity checks** that are awkward to fully encode as unit tests.

Checks included:
- `W` eigenvalues and the PSD requirement (MUDAG expects PSD `W`).
- MUDAG inner-loop length `K` and the quantities it depends on.
- Basic convergence trends and correct cost accounting (`mix_rounds`, `grad_evals_per_node`).

Notes:
- Use `lazy=0.5` (i.e. `W <- (W+I)/2`) to match the archived MATLAB workflow that uses a PSD mixing matrix.
- The residual `avg_sq_dist_to_x_star_all_nodes` is the MATLAB-style metric $(1/n)\|X-\mathbf{1}x^{*\top}\|_F^2$ (squared tolerance).

In [11]:
import sys
from pathlib import Path

import numpy as np

# Ensure repo root is on sys.path so `import research...` works no matter
# which working directory VS Code/Jupyter chooses for this notebook.
_here = Path().resolve()
_repo_root = None
for p in [_here, *_here.parents]:
    if (p / "pyproject.toml").exists():
        _repo_root = p
        break
if _repo_root is not None and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from research.code.distopt.generators import (
    path_adjacency,
    make_graph_from_adjacency,
    make_shared_eigenbasis_problem,
 )
from research.code.distopt.algorithms import DGD, MUDAG
from research.code.distopt.runner import (
    run_experiment,
    MaxIters,
    TargetAvgSqDistToXStarAllNodes,
 )


In [12]:
# 1) Build a small graph and confirm PSD W via eigenvalues
n = 10
adj = path_adjacency(n)
graph = make_graph_from_adjacency(adj, lazy=0.5)
gstats = graph.ensure_stats()

print('connected:', gstats.connected)
print('lambda2_W:', gstats.lambda2_W)
print('min eig(W):', float(np.min(gstats.eigvals_W)))
print('max eig(W):', float(np.max(gstats.eigvals_W)))

connected: True
lambda2_W: 0.9836855054317178
min eig(W): 0.3496478279016157
max eig(W): 1.0


In [13]:
# 2) Build a toy quadratic problem and inspect the constants used in MUDAG
problem = make_shared_eigenbasis_problem(graph, d=5, mu=1.0, L=10.0, seed=0)
pstats = problem.ensure_stats()

print('kappa_g:', pstats.kappa_g)
print('L_g:', pstats.L_g, 'mu_g:', pstats.mu_g)
print('L_l (M):', pstats.L_l, ' => M/L:', pstats.L_l / pstats.L_g)

kappa_g: 9.230367088281866
L_g: 9.755995310679067 mu_g: 1.0569455382835744
L_l (M): 10.000000000000009  => M/L: 1.0250107427843729


In [14]:
# 3) Run MUDAG briefly and verify cost accounting + residual trends
# Note: depending on the instance, very small c_K can yield a too-short inner loop (small K)
# and may diverge. For this validation run we use a slightly larger c_K to stay stable.
alg = MUDAG(c_K=0.5)
res = run_experiment(
    problem,
    alg,
    stop=[MaxIters(30)],
    log_every=1,
)

print('Computed K:', int(round(res.history[0]['mudag_K'])))
print('Final mix_rounds:', int(res.final['mix_rounds']))
print('Final grad_evals_per_node:', int(res.final['grad_evals_per_node']))
print('Final avg_sq_dist:', res.history[-1]['avg_sq_dist_to_x_star_all_nodes'])


Computed K: 9
Final mix_rounds: 300
Final grad_evals_per_node: 59
Final avg_sq_dist: 0.08291364144707038


In [15]:
# 4) Optional: stop on MATLAB-style residual (squared tolerance)
tol = 1e-4  # squared tolerance
res_stop = run_experiment(
    problem,
    MUDAG(c_K=0.5),
    stop=[MaxIters(200), TargetAvgSqDistToXStarAllNodes(tol)],
    log_every=1,
 )
print('Stopped at t=', res_stop.final['t'], 'mix=', res_stop.final['mix_rounds'])
print('Last residual:', res_stop.history[-1]['avg_sq_dist_to_x_star_all_nodes'])


Stopped at t= 179 mix= 1790
Last residual: 9.94021854001376e-05


In [16]:
# 5) Compare against a simple baseline (DGD) by mix rounds
# Note: DGD uses 1 mix/iter, while MUDAG uses K+1 mixes/outer-iter.
dgd = DGD(alpha=0.05)
res_dgd = run_experiment(problem, dgd, stop=[MaxIters(200)], log_every=10)

print('DGD final mix_rounds:', int(res_dgd.final['mix_rounds']), 'avg_sq_dist:', res_dgd.history[-1]['avg_sq_dist_to_x_star_all_nodes'])
print('MUDAG final mix_rounds:', int(res.final['mix_rounds']), 'avg_sq_dist:', res.history[-1]['avg_sq_dist_to_x_star_all_nodes'])

DGD final mix_rounds: 200 avg_sq_dist: 0.11626804678212492
MUDAG final mix_rounds: 300 avg_sq_dist: 0.08291364144707038
